## Tema 4 - Listado 1 - Ejercicio 1

Ejercicio para entrenar un modelo simple de red neuronal utilizando la biblioteca Keras de Tensorflow.

1) En  este  primer  ejercicio  no  se  va  a  realizar  ningún  tipo  de  preprocesamiento  del 
texto. Las etiquetas (labels) serán 1 para la clase positiva y 0 para la clase negativa. 
Se define una longitud para los textos = 10. Se define un tamaño de vocabulario = 50. 
Hay que ir siguiendo los pasos que se indican a continuación: 
2) Construir  el  vocabulario  (los  tokens  diferentes  de  los  textos  de  entrada).  Mostrar 
los tokens del vocabulario con su índice.  

3) Construir la matriz de documentos-términos con el tamaño máximo elegido, donde 
el peso de cada término es el índice en el vocabulario (si no está se le pone valor 0). 

4) Una vez preprocesado el texto, hay que definir el modelo.  Se va a utilizar la clase 
Sequential, que permite añadir diferentes capas de forma secuencial.  

5) La  primera  capa  del  modelo  es  la  capa  de  embeddings,  que  se  encargará  de 
representar  cada  texto  como  una  matriz  de  vectores.  La  matriz  tendrá  como 
número  de  filas  el  valor  de  la  variable  max_length,  es  decir,  la  longitud  de  las 
secuencias  de  entrada.  Como  número  de  columnas,  tendremos  que  decidir  que 
dimensión  queremos  utilizar  para  representar  los  vectores  de  los  tokens.  En  este 
ejemplo,  vamos  a  utilizar  un  tamaño  de  vector  pequeño  (vector_size  =  8), 
porque estamos con un ejemplo de juguete (las dimensiones que se suelen utilizar 
más son 100, 200 o 300). 

La capa Embedding lo que hará será inicializar una matriz por cada texto. Como se 
ha dicho antes la matriz, tendrá una dimensión de max_length x vector_size. 
En este ejercicio, la matriz se inicializa con pesos aleatorios, pero se podría inicializar 
la matriz a partir de un modelo pre-entrenado de word embeddings (lo veremos en 
otro ejercicio). 

Hay que añadir una capa densa para la salida. En este caso, al ser una salida binaria 
como función de activación podemos utilizar sigmoid. Devolverá una probabilidad, 
que si es cercana a 1 entonces la capa de salida devolverá 1, 0 en otro caso.

## Carga de datos

El conjunto de datos son frases sencillas anotadas manualmente con su sentido positivo o negativo. 

* "Estoy un poco harto del día a día, nada mejora" -> Negativo
* "Hoy es un buen día" -> Positivo
* "No se te ve satisfecho con el trabajo" -> Negativo
* "Este paisaje es hermoso y bonito" -> Positivo


In [9]:
sentences = ['Estoy un poco harto del día a día , nada mejora',
             'Hoy es un buen día',
             'No se te ve satisfecho con el trabajo',
             'Este paisaje es hermoso y bonito']

# 1: positivo, 0: negativo
labels = [0,1,0,1]


Hay que preparar los datos de entrenamiento:

* Longitud de las secuencias de texto = 10
* Tamaño del vocabulario = 50
  

In [20]:
import spacy
import spacy.cli
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Cargar el modelo de español de Spacy
nlp = spacy.load('es_core_news_sm')

# Para preparar el vocabulario
def prepare_vocabulary(corpus, vocab_size):    
   
    # Crear un diccionario para mapear tokens a índices
    token_to_index = {}
    current_index = 1 # importante empezar en 1, 0 se usa para padding

    # Primera pasada para construir vocabulario
    for sentence in corpus:
        doc = nlp(sentence)
        for token in doc:
            if token.text not in token_to_index and current_index < vocab_size:
                token_to_index[token.text] = current_index
                current_index += 1
    
    return token_to_index


# representación one hot
# Codificamos la frase con el índice del boc (si no está se pone id = 0), luego aplicamos padding y trunc (pad_secuences) para q tengan el mismo n dim
def prepare_sentences(corpus, vocabulary, max_length):
    encoded_sentences = []
    
    # Segunda pasada para codificar oraciones
    for sentence in corpus:
        doc = nlp(sentence)
        encoded_sentence = []
        for token in doc:
            # Si el token está en nuestro vocabulario, usar su índice
            if token.text in vocabulary:
                encoded_sentence.append(vocabulary[token.text])
            # Si no está, usar el índice 0 (desconocido)
            else:
                encoded_sentence.append(0)
        encoded_sentences.append(encoded_sentence)

    # Hacer padding de las secuencias
    prepared_sentences = pad_sequences(encoded_sentences, maxlen=max_length, padding='post', truncating='post')
    print("Oraciones originales(",len(corpus),"):")
    print(corpus)  
    print("Oraciones procesadas(",len(prepared_sentences),"):")
    print(prepared_sentences)    
    return prepared_sentences

# Uso
vocab_size = 50
max_length = 10
vocabulary_train = prepare_vocabulary(sentences,vocab_size)
print("\nVocabulario (",len(vocabulary_train),"):")
print(vocabulary_train)
prepared_sentences = prepare_sentences(sentences, vocabulary_train, max_length)


Vocabulario ( 26 ):
{'Estoy': 1, 'un': 2, 'poco': 3, 'harto': 4, 'del': 5, 'día': 6, 'a': 7, ',': 8, 'nada': 9, 'mejora': 10, 'Hoy': 11, 'es': 12, 'buen': 13, 'No': 14, 'se': 15, 'te': 16, 've': 17, 'satisfecho': 18, 'con': 19, 'el': 20, 'trabajo': 21, 'Este': 22, 'paisaje': 23, 'hermoso': 24, 'y': 25, 'bonito': 26}
Oraciones originales( 4 ):
['Estoy un poco harto del día a día , nada mejora', 'Hoy es un buen día', 'No se te ve satisfecho con el trabajo', 'Este paisaje es hermoso y bonito']
Oraciones procesadas( 4 ):
[[ 1  2  3  4  5  6  7  6  8  9]
 [11 12  2 13  6  0  0  0  0  0]
 [14 15 16 17 18 19 20 21  0  0]
 [22 23 12 24 25 26  0  0  0  0]]


## CNN

In [11]:
import tensorflow as tf
tf.__version__

'2.19.0'

## Configuramos el modelo

In [13]:
from keras.models import Sequential
from keras.layers import Flatten, Dense, Embedding, Conv1D, MaxPooling1D

# Crear una red secuencial para el modelo
model = Sequential()

# Añadir una capa inicial de embedding que transforma los índices de palabras en vectores densos
vector_size = 8
model.add(Embedding(vocab_size, vector_size))

# Añadir una capa de aplanado (Flatten) para aplanar la entrada, convirtiendo los datos multidimensionales en un vector unidimensional
model.add(Flatten())

# Añadir una capa densa con 1 neurona para una salida binaria con una función de activación sigmoid para clasificación binaria. Devuelve una probabilidad.
# Si la probabilidad es cercana a 1, la capa devuelve 1, y 0 en otro caso. 
model.add(Dense(1, activation='sigmoid'))

print("Red diseñada correctamente")


Red diseñada correctamente


## Compilamos del modelo

In [21]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary of the model
model.build(input_shape=(None, max_length)) 
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 10, 8)          │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            81 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 481 (1.88 KB)

 Trainable params: 481 (1.88 KB)

 Non-trainable params: 0 (0.00 B)

## Entrenamos el modelo

In [22]:
from sklearn.model_selection import train_test_split
import numpy as np

# Convertir a NumPy arrays para asegurar compatibilidad y rendimiento
prepared_sentences = np.array(prepared_sentences)
labels = np.array(labels)

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(prepared_sentences, labels, test_size=0.2, random_state=42)

# Entrenar el modelo
batch_size = 32
epochs = 5
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), batch_size=batch_size, epochs=epochs)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 645ms/step - accuracy: 1.0000 - loss: 0.6609 - val_accuracy: 1.0000 - val_loss: 0.6682
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 1.0000 - loss: 0.6558 - val_accuracy: 1.0000 - val_loss: 0.6673
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 1.0000 - loss: 0.6507 - val_accuracy: 1.0000 - val_loss: 0.6664
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 1.0000 - loss: 0.6457 - val_accuracy: 1.0000 - val_loss: 0.6656
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 1.0000 - loss: 0.6406 - val_accuracy: 1.0000 - val_loss: 0.6648


## Evaluar el modelo

Vamos a evaluarlo sobre el conjunto test. En primer lugar, vamos a obtener las métricas loss y accuracy en dicho conjunto (que no ha sido utilizado en ninguna fase del entrenamiento).


In [19]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Conjunto más amplio de frases de prueba
test_sentences = [
    "No fui al estreno de la película porque nadie me quería acompañar",
    "Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio",
    "Se nos está volviendo costumbre del domingo por la noche, ver el episodio anterior de SNL y eso me hace recibir el lunes con mejor humor",
    "Al final decidí no ir al cine porque estaba cansada",
    "Todo es maravilloso y formidable, muy bonito"
]

# Preparar los datos
voc_test= prepare_vocabulary(test_sentences,vocab_size=vocab_size)
prepared_test= prepare_sentences(test_sentences, voc_test, max_length)

# Para recordar en la salida cuál era el vocabulario
print("\nVocabulario (",len(vocabulary_train),"):")
print(vocabulary_train)

prepared_test = prepare_sentences(test_sentences, vocabulary_train, max_length)

# Realizar predicciones
predictions = model.predict(prepared_test)

# Interpretar las predicciones con más detalle
print("Predicciones detalladas:")
for i, sentence in enumerate(test_sentences):
    pred = predictions[i][0]
    sentiment = "Positivo" if pred > 0.5 else "Negativo"
    print(f"\nTexto: {sentence}")
    print(f"Predicción numérica: {pred:.4f}")
    print(f"Sentimiento predicho: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 1.0000 - loss: 0.6690
Accuracy: 100.00%
Oraciones originales( 5 ):
['No fui al estreno de la película porque nadie me quería acompañar', 'Envidio de buena manera a los que tienen la oportunidad de ir mañana al estadio', 'Se nos está volviendo costumbre del domingo por la noche, ver el episodio anterior de SNL y eso me hace recibir el lunes con mejor humor', 'Al final decidí no ir al cine porque estaba cansada', 'Todo es maravilloso y formidable, muy bonito']
Oraciones procesadas( 5 ):
[[ 1  2  3  4  5  6  7  8  9 10]
 [13  5 14 15 16 17 18 19  6 20]
 [24 25 26 27 28 29 30 31  6 32]
 [47 48 49  0 21  3  0  8  0  0]
 [ 0  0  0 39  0 33  0  0  0  0]]

Vocabulario ( 26 ):
{'Estoy': 1, 'un': 2, 'poco': 3, 'harto': 4, 'del': 5, 'día': 6, 'a': 7, ',': 8, 'nada': 9, 'mejora': 10, 'Hoy': 11, 'es': 12, 'buen': 13, 'No': 14, 'se': 15, 'te': 16, 've': 17, 'satisfecho': 18, 'con': 19, 'el': 20, 'trabajo': 21, 'Este': 22, 'paisaje': 23, 'hermoso': 24